# LRE: training on Google Colab (Phase 5)

Trains the learned parts of the Legacy Land Record Engine on a free GPU and evaluates them against
the Tesseract baseline (report Table 11).

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.

| Step | What | Rough time on a T4 |
|---|---|---|
| 1-3 | GPU check, Drive, code, packages | 3 min |
| 4 | regenerate SynJama (1,000 pages) + line sets, saved to Drive | 40-50 min, first session only |
| 5 | CRNN stage B, 15 epochs (checkpoint saved to Drive every epoch) | 1.5-3 h |
| 6 | optional: U-Net, CRF, baseline on this data | 10-40 min |
| 7 | evaluate Proposed A and B | 20-30 min |
| 8 | Table 11 + copy everything to Drive | 1 min |

If Colab disconnects during step 5, rerun steps 1-4 (step 4 then just unzips) and continue training with
`--init` pointing at the checkpoint in Drive.

In [ ]:
# 1. GPU and Python
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

In [ ]:
# 2. Google Drive: everything that must survive the session goes to DRIVE
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/lre'
os.makedirs(f'{DRIVE}/models', exist_ok=True)
os.makedirs(f'{DRIVE}/results', exist_ok=True)

### 3. Get the code

**Option A (zip, no GitHub needed).** On your Mac, in Terminal:

```
cd ~/Documents
zip -r lre_code.zip AI_project -x "AI_project/.venv/*" "AI_project/data/synjama/*" "AI_project/data/lines/*" "AI_project/data/interim/*" "AI_project/*.docx" "AI_project/results/lre.db"
```

then upload `lre_code.zip` to `My Drive/lre/` and run the next cell.

**Option B (later, once you push the repo).** Replace the unzip line with `git clone`.

In [ ]:
!rm -rf /content/AI_project
!unzip -q "{DRIVE}/lre_code.zip" -d /content
# Option B: !git clone https://github.com/<you>/<repo>.git /content/AI_project
%cd /content/AI_project

# Colab already has CUDA torch, numpy, pandas, scikit-learn, scikit-image, opencv, Pillow; add the rest
!apt-get -qq install -y libfribidi0 > /dev/null
!pip install -q -r requirements-colab.txt
!pip install -q -e . --no-deps
# smaller page files on Colab (the pipeline converts to grayscale anyway)
!sed -i 's/^  image_format: png/  image_format: jpg/' config.yaml

from PIL import features
import torch
print('Devanagari shaping (raqm):', features.check('raqm'), '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())
!lre fonts && lre version

### 4. Data

Regenerates SynJama with the same seed and settings as the local set (same holdings, registers and
labels; pixels can differ slightly with Colab's font libraries), then cuts the CRNN line sets.
Everything is zipped to Drive, so later sessions just unzip.

If the shaping check above printed `False`, run `!apt-get install -y libraqm0` and restart the runtime.

In [ ]:
# Pages and lines are zipped to Drive separately, so a new session (or a dropped runtime) reuses them.
PAGES_ZIP, LINES_ZIP = f'{DRIVE}/synjama_pages.zip', f'{DRIVE}/lines.zip'
if os.path.exists('data/synjama/test/manifest.json'):
    print('pages already rendered in this session')
elif os.path.exists(PAGES_ZIP):
    !unzip -q -o "{PAGES_ZIP}" -d /content/AI_project
else:
    !lre synth --pages 1000 --out data/synjama --workers 2          # ~40 min
if not os.path.exists(PAGES_ZIP):
    !zip -q -0 -r "{PAGES_ZIP}" data/synjama                        # -0: pages are already JPEG
if os.path.exists(LINES_ZIP):
    !unzip -q -o "{LINES_ZIP}" -d /content/AI_project
else:
    !pkill -f "lre build-line[s]"; sleep 2                          # leftovers of an interrupted run
    !lre build-lines --split train --limit 600 --workers 2          # ~5 s/page on Colab's 2 CPUs: ~50 min, ~72k lines
    !lre build-lines --split val --workers 2                        # ~8 min
    !zip -q -0 -r "{LINES_ZIP}" data/lines
!du -sh data/synjama data/lines

### 5. Train the CRNN, stage B (report 5.3.2, Table 6)

15 epochs, AdamW + one-cycle, batch 32, mixed precision, early stop after 5 epochs without a
validation CER gain. The best checkpoint and its model card go straight to Drive.

To continue after a disconnect: `!lre train-crnn --stage B --epochs 8 --init {CRNN} --out {DRIVE}/models/crnn_B2.pt`

Stage A (IIIT-HW-Dev + IAM) and stage C (HandFilled) need those datasets packed the same way;
that converter is not written yet.

In [ ]:
CRNN = f'{DRIVE}/models/crnn_B.pt'
# The best epoch is saved to Drive as it trains; --max-minutes stops in time on the free runtime
!lre train-crnn --stage B --epochs 12 --max-minutes 110 --out "{CRNN}"
!cat "{DRIVE}/models/crnn_B.card.json" | head -40

### 6. Optional

The U-Net and CRF were already trained on the Mac and come with the code zip (`models/`).
Retrain them here only if you want them fitted to the Colab-generated data.
Re-running the baseline here makes Table 11 compare all systems on exactly the same pages.

In [ ]:
# !lre train-unet --out models/unet_v1.pt
# !lre train-crf --out models/crf_owner_v1.joblib
!apt-get -qq install -y tesseract-ocr > /dev/null && lre tessdata
!lre evaluate --stage baseline --split test --workers 2

### 7. Evaluate (report 7.1)

Proposed A = grid + CRNN + field-aware decoding + CRF + rules.
Proposed B = A + gradient-boosted contradiction scorer (trained on the train-page flags) + calibrated
routing (theta and per-field thresholds fitted on val), applied once to test.

In [ ]:
UNET = 'models/unet_v1.pt'
!lre evaluate --stage proposed --split train --limit 150 --crnn "{CRNN}" --unet {UNET}
!lre evaluate --stage proposed --split val --crnn "{CRNN}" --unet {UNET}
!lre evaluate --stage proposed --split test --crnn "{CRNN}" --unet {UNET}
!lre evaluate --stage proposed_b --split test

In [ ]:
# 8. Table 11 and copy everything to Drive
import json, pandas as pd
rows = []
for name in ['baseline', 'proposed_a', 'proposed_b']:
    p = f'results/{name}_metrics.json'
    if os.path.exists(p) and 'test' in json.load(open(p)):
        m = json.load(open(p))['test']
        c = m.get('contradiction', {})
        rows.append([name, m['mean_field_accuracy'], m['name_cer'], m['numeric_field_accuracy'],
                     c.get('precision'), c.get('recall'), m['review_share']])
display(pd.DataFrame(rows, columns=['system', 'mean field acc.', 'name CER', 'numeric field acc.',
                                    'contr. precision', 'contr. recall', 'review share']).round(4))
!cp -r results/. "{DRIVE}/results/"
!cp models/*.pt models/*.joblib models/*.json "{DRIVE}/models/" 2>/dev/null
!ls -la "{DRIVE}/models" "{DRIVE}/results"

### Back on the Mac

Copy from `My Drive/lre/` into the project:

* `models/crnn_B.pt`, `models/gbm_v1.joblib`, `models/calibrators_v1.joblib` (and the `.card.json` files) -> `AI_project/models/`
* `results/*.json` and `results/tables/` -> `AI_project/results/`

Then the full proposed pipeline runs locally:

```
lre run --village V000 --pages data/sample/pages --register data/sample/register.csv --extractor crnn --crnn models/crnn_B.pt
streamlit run app/reviewer.py
```